In [ ]:
import pandas as pd
import re # 정규식
import smtplib # 이메일을 전송할 때 사용하는 SMTP 프로토콜 관련 표준 라이브러리

from getpass import getpass # 앱 비밀번호 보안 입력창
from email.message import EmailMessage  # 이메일의 본문, 제목, 첨부파일 등의 규격화

input_file = "사내SNS댓글_데이터.csv"
df = pd.read_csv(input_file)

df = df.dropna(subset=["comment_id"]) # subset으로 해당 열만 확인 # 말 그대로 N/A인 값을 지움 (이후에도 마찬가지)
df = df.dropna(subset=["comment_text"])
df = df.drop_duplicates(subset=["comment_id"]) # 중복된 데이터를 처음 값만 남기고 나머지는 모두 제거 (keep="first"는 기본값이므로 굳이 안 써도 됨)

df["author"] = df["author"].fillna("알 수 없음")

def clean_text(text):
    text = re.sub(r"<.*?>", "", text) #  HTML 태그 모두 제거       
    text = re.sub(r"[\n\t]", " ", text) # 줄바꿈, 탭은 일반 공백 한 칸(" ")으로     
    text = re.sub(r"\s+", " ", text) # 하나 이상의 공백은 역시 일반 공백 한 칸(" ")으로        
    
    return text.strip() # 맨 앞과 맨 뒤의 공백 제거                     

df["comment_text"] = df["comment_text"].apply(clean_text)
df = df[df["comment_text"] != ""] # 공백만 있던 댓글이 clean_text 후 빈 문자열이 된 경우 제거

def get_number(value):
    number = re.sub(r"\D", "", str(value)) # 숫자가 아닌 모든 문자는 빈 문자열("")로 '지우기'  
    return int(number)

df["like_count"] = df["like_count"].apply(get_number)
df["reply_count"] = df["reply_count"].apply(get_number)

df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce", format="mixed") # 변환할 수 없는 값은 NaT(날짜/시간형 결측값)로 처리 # format="mixed" : 행마다 서로 다른 날짜 표기 형식을 자동으로 해석
df = df.dropna(subset=["published_at"])

output_file = "전처리_댓글.csv"

df.to_csv(output_file, index=False, encoding="utf-8-sig")

print("전처리 완료 :", len(df), "건") # 마지막까지 살아남은 데이터의 행 개수 출력
print("CSV 저장 완료 :", output_file)

sender = os.getenv("SENDER_EMAIL", "")
receiver = os.getenv("RECEIVER_EMAIL", sender)
password = getpass("Gmail 앱 비밀번호 : ")

msg = EmailMessage() # 이메일 작성을 위한 비어있는 객체 생성

msg["Subject"] = "[댓글수집] SNS 댓글 전처리 결과"
msg["From"] = sender
msg["To"] = receiver

msg.set_content(
    f"""안녕하세요.

SNS 댓글 전처리 결과를 전달드립니다.

처리된 댓글 수 : {len(df)}건

첨부파일을 확인해 주세요.
"""
)

with open(output_file, "rb") as f: # csv 파일을 바이너리 읽기("rb")모드로 안전하게 불러오기

    # 이메일에 파일 데이터 추가
    msg.add_attachment(
        f.read(), # 파일 데이터 통째로 읽기
        maintype="text", # 파일 유형 : 일반 텍스트
        subtype="csv", # 파일 형식 : .csv
        filename=output_file # 수신자 이메일에서 보일 첨부파일명
    )

with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp: # 보안 연결(SSL)을 사용해서 Gmail 전송 서버(smtp.gmail.com, 465 포트)에 접속
    smtp.login(sender, password)
    smtp.send_message(msg)

print("이메일 전송 완료")